# 05 Silver — DLT Integration Pipeline

**Laag:** Integration (Silver)  
**Catalog:** `DEMO`  
**Schema:** `INTEGRATION`  
**Pipeline type:** Lakeflow Declarative Pipeline (DLT)

## Wat deze pipeline doet

Leest uit Bronze CDF (`readChangeFeed`) en levert vijf gecleande tabellen in `INTEGRATION`:

| Tabel | Type | Inhoud |
|---|---|---|
| `order_header` | Streaming Table | Gecleaned `order_header` — rijen die alle drop-regels passeren |
| `order_header_quarantine` | Streaming Table | Rijen die één of meer drop-regels schenden + `failed_rules` ARRAY<STRING> |
| `order_detail` | Streaming Table | Gecleaned `order_detail` *(scaffold — regels in volgende slice)* |
| `order_detail_quarantine` | Streaming Table | Schendende detail-rijen *(scaffold)* |
| `sales_line` | Materialised View | Geïntegreerde view: `order_header ⨝ order_detail` op `order_id` |

## Drie ernstniveaus (CONTEXT.md §7)

| Niveau | DLT-mechanisme | Effect |
|---|---|---|
| `fail` | `@dlt.expect_all_or_fail` | Pipeline halt op schending — voor invariants die nooit mogen falen |
| `drop` | filter + `_quarantine`-tabel | Rij wordt geroute naar `_quarantine` met `failed_rules` ARRAY<STRING> |
| `warn` | `@dlt.expect_all` | Rij blijft in cleansed; schending wordt geteld in DLT-events |

## Quarantine-patroon (Pattern A)

Beide tabellen lezen vanuit dezelfde Bronze CDF via een gedeeld predicate:

```python
# Cleansed — rijen die alle drop-regels passeren
.filter(clean_predicate)

# Quarantine — inverse + welke regels gefaald zijn
.filter(f"NOT ({clean_predicate})")
.withColumn("failed_rules", build_failed_rules_expr(DROP_RULES))
```

DLT graph view toont twee nodes (`order_header` en `order_header_quarantine`) beide
gevoed vanuit dezelfde Bronze CDF-bron.

In [ ]:
# ── DLT-imports ────────────────────────────────────────────────────────────────
import dlt
from pyspark.sql import functions as F
from pyspark.sql.types import DecimalType, StringType

# ── Rule engine — pure-Python helper, no Spark dependency in the API ───────────
# Imported from integration/rule_engine.py (same directory as this notebook).
# The DLT cluster adds the notebook's directory to sys.path automatically.
from rule_engine import build_clean_predicate, build_failed_rules_expr

## Parameters

The DLT pipeline configuration (catalog, Bronze source schema) is supplied via DLT
pipeline parameters rather than `dbutils.widgets` — DLT notebooks run in a managed
execution context where widgets are not available at definition time.

Set these under **Pipeline settings → Configuration** in the DLT UI, or via
`resources/dlt_integration.yml`:

| Key | Default | Description |
|---|---|---|
| `catalog` | `DEMO` | Unity Catalog catalog name |
| `bronze_schema` | `STAGING_AZURESTORAGE` | Schema where Bronze tables live |

In [ ]:
# Read DLT pipeline configuration parameters with sensible defaults.
catalog       = spark.conf.get("catalog",       "DEMO")
bronze_schema = spark.conf.get("bronze_schema", "STAGING_AZURESTORAGE")

# Fully-qualified Bronze source table names.
BRONZE_ORDER_HEADER = f"{catalog}.{bronze_schema}.order_header"
BRONZE_ORDER_DETAIL = f"{catalog}.{bronze_schema}.order_detail"

## Rule definitions — `order_header`

Rule sets per severity level (see `CONTEXT.md` §7 for the full specification).

| Rule | Severity |
|---|---|
| `order_id IS NOT NULL` | fail |
| `order_ts IS NOT NULL` | drop |
| `customer_id IS NOT NULL` | drop |
| `order_currency IS NOT NULL` | drop |
| `order_total >= 0` | drop |
| `order_amount >= 0` | drop |
| `truck_id IS NOT NULL` | warn |
| `location_id IS NOT NULL` | warn |
| `shift_start_time <= shift_end_time` | warn |

In [ ]:
# ── order_header rule sets ─────────────────────────────────────────────────────

# FAIL rules — pipeline halts on violation.  Applied via @dlt.expect_all_or_fail.
ORDER_HEADER_FAIL_RULES: dict[str, str] = {
    "order_id_not_null": "order_id IS NOT NULL",
}

# DROP rules — rows routed to _quarantine on violation.
# Used to build the shared clean_predicate that splits cleansed vs quarantine.
ORDER_HEADER_DROP_RULES: dict[str, str] = {
    "order_ts_not_null":      "order_ts IS NOT NULL",
    "customer_id_not_null":   "customer_id IS NOT NULL",
    "order_currency_not_null":"order_currency IS NOT NULL",
    "order_total_positive":   "order_total >= 0",
    "order_amount_positive":  "order_amount >= 0",
}

# WARN rules — rows stay in cleansed; violations counted in DLT events.
# Applied via @dlt.expect_all.
ORDER_HEADER_WARN_RULES: dict[str, str] = {
    "truck_id_not_null":              "truck_id IS NOT NULL",
    "location_id_not_null":           "location_id IS NOT NULL",
    "shift_start_before_end":         "shift_start_time <= shift_end_time",
}

# Build the shared predicates used for Pattern A quarantine routing.
OH_CLEAN_PREDICATE   = build_clean_predicate(ORDER_HEADER_DROP_RULES)
OH_FAILED_RULES_EXPR = build_failed_rules_expr(ORDER_HEADER_DROP_RULES)

## Bronze CDF helper

Both `order_header` and `order_header_quarantine` read from the same Bronze CDF
stream.  A helper function returns the cleaned (type-fixed, snake_case) Bronze
stream so the two DLT table definitions share exactly one read path — the DLT
engine is responsible for materialising the two filtered views from it.

In [ ]:
def _read_bronze_order_header():
    """Return a streaming DataFrame from Bronze order_header CDF.

    Applies all type-fixes and snake_case renames defined in CONTEXT.md §7.
    The returned DataFrame is filtered *neither* for cleansed nor quarantine —
    callers apply the appropriate filter.

    Type-fix reference (CONTEXT.md §7 — Bronze → Silver):
      SERVED_TS              StringType  → TimestampType  (yyyy-MM-dd HH:mm:ss)
      ORDER_TAX_AMOUNT       StringType  → DecimalType(38,4)
      ORDER_DISCOUNT_AMOUNT  StringType  → DecimalType(38,4)
      LOCATION_ID            DoubleType  → DecimalType(38,0)  (IDs are not Doubles)
      DISCOUNT_ID            StringType  → DecimalType(38,0) nullable
      SHIFT_START_TIME       IntegerType → StringType 'HH:mm:ss'  (millis→time)
      SHIFT_END_TIME         IntegerType → StringType 'HH:mm:ss'  (millis→time)
    """
    raw = (
        spark.readStream
        .option("readChangeFeed", "true")
        .table(BRONZE_ORDER_HEADER)
    )

    return (
        raw
        # ── snake_case renames + type-fixes ───────────────────────────────────
        .withColumnRenamed("ORDER_ID",               "order_id")
        .withColumnRenamed("TRUCK_ID",               "truck_id")
        .withColumn(
            "location_id",
            F.col("LOCATION_ID").cast(DecimalType(38, 0))
        )
        .drop("LOCATION_ID")
        .withColumnRenamed("CUSTOMER_ID",            "customer_id")
        .withColumn(
            "discount_id",
            F.col("DISCOUNT_ID").cast(DecimalType(38, 0))
        )
        .drop("DISCOUNT_ID")
        .withColumnRenamed("SHIFT_ID",               "shift_id")
        .withColumn(
            # millis since midnight → 'HH:mm:ss'
            "shift_start_time",
            F.date_format(
                F.to_timestamp(F.col("SHIFT_START_TIME") / 1000),
                "HH:mm:ss"
            )
        )
        .drop("SHIFT_START_TIME")
        .withColumn(
            "shift_end_time",
            F.date_format(
                F.to_timestamp(F.col("SHIFT_END_TIME") / 1000),
                "HH:mm:ss"
            )
        )
        .drop("SHIFT_END_TIME")
        .withColumnRenamed("ORDER_CHANNEL",          "order_channel")
        .withColumnRenamed("ORDER_TS",               "order_ts")
        .withColumn(
            "served_ts",
            F.to_timestamp(F.col("SERVED_TS"), "yyyy-MM-dd HH:mm:ss")
        )
        .drop("SERVED_TS")
        .withColumnRenamed("ORDER_CURRENCY",         "order_currency")
        .withColumnRenamed("ORDER_AMOUNT",           "order_amount")
        .withColumn(
            "order_tax_amount",
            F.col("ORDER_TAX_AMOUNT").cast(DecimalType(38, 4))
        )
        .drop("ORDER_TAX_AMOUNT")
        .withColumn(
            "order_discount_amount",
            F.col("ORDER_DISCOUNT_AMOUNT").cast(DecimalType(38, 4))
        )
        .drop("ORDER_DISCOUNT_AMOUNT")
        .withColumnRenamed("ORDER_TOTAL",            "order_total")
        # ── keep Bronze audit columns for lineage ─────────────────────────────
        .withColumnRenamed("_ingestion_timestamp",   "_ingestion_timestamp")
        .withColumnRenamed("_source_system",         "_source_system")
        .withColumnRenamed("_source_file",           "_source_file")
        .withColumnRenamed("_last_modified",         "_last_modified")
        .withColumnRenamed("_pipeline_run_id",       "_pipeline_run_id")
    )

## `INTEGRATION.order_header` — Cleansed Streaming Table

Receives rows that pass **all** drop rules.  
Warn rules are applied as DLT Expectations — violations are counted but rows stay.
Fail rules halt the pipeline if violated.

In [ ]:
@dlt.expect_all_or_fail(ORDER_HEADER_FAIL_RULES)
@dlt.expect_all(ORDER_HEADER_WARN_RULES)
@dlt.table(
    name="order_header",
    comment="Gecleaned order_header — rijen die alle drop-regels passeren. "
            "Fail-regels halteren de pipeline; warn-regels tellen schendingen in DLT-events.",
    table_properties={
        "quality": "silver",
        "pipelines.autoOptimize.managed": "true",
    },
)
def order_header():
    """Cleansed order_header Streaming Table.

    Pattern A: filter on the shared clean_predicate keeps only rows that pass
    every DROP rule.  The inverse lands in order_header_quarantine.
    FAIL expectations are checked *after* the drop-filter, so a NULL order_id
    in a row that already violates a drop rule goes to quarantine first — the
    fail guard covers only rows that make it through the drop filter.
    """
    return _read_bronze_order_header().filter(OH_CLEAN_PREDICATE)

## `INTEGRATION.order_header_quarantine` — Quarantine Streaming Table

Receives rows that fail **at least one** drop rule.  
The `failed_rules` ARRAY<STRING> column lists every rule name the row violated,
enabling analyst triage:

```sql
SELECT *
FROM   INTEGRATION.order_header_quarantine
WHERE  array_contains(failed_rules, 'order_total_positive');
```

In [ ]:
@dlt.table(
    name="order_header_quarantine",
    comment="order_header rijen die één of meer drop-regels schenden. "
            "failed_rules ARRAY<STRING> lijst elke geschonden regel-naam.",
    table_properties={
        "quality": "quarantine",
        "pipelines.autoOptimize.managed": "true",
    },
)
def order_header_quarantine():
    """Quarantine Streaming Table — Pattern A inverse filter.

    Reads the same Bronze CDF stream as order_header but keeps only rows that
    fail the clean_predicate.  Adds failed_rules ARRAY<STRING> via rule_engine.
    """
    return (
        _read_bronze_order_header()
        .filter(f"NOT ({OH_CLEAN_PREDICATE})")
        .withColumn("failed_rules", OH_FAILED_RULES_EXPR)
    )

## `INTEGRATION.order_detail` — Scaffold (next slice)

Bronze CDF reader and type-fix helper for `order_detail`.  
Drop/fail/warn rules will be added in the `order_detail` quarantine slice.

In [ ]:
def _read_bronze_order_detail():
    """Return a streaming DataFrame from Bronze order_detail CDF.

    Type-fixes applied (CONTEXT.md §7):
      ORDER_ITEM_DISCOUNT_AMOUNT  StringType → DecimalType(38,4)
    All other columns are renamed snake_case only.
    """
    raw = (
        spark.readStream
        .option("readChangeFeed", "true")
        .table(BRONZE_ORDER_DETAIL)
    )

    return (
        raw
        .withColumnRenamed("ORDER_DETAIL_ID",            "order_detail_id")
        .withColumnRenamed("ORDER_ID",                   "order_id")
        .withColumnRenamed("MENU_ITEM_ID",               "menu_item_id")
        .withColumn(
            "discount_id",
            F.col("DISCOUNT_ID").cast(DecimalType(38, 0))
        )
        .drop("DISCOUNT_ID")
        .withColumnRenamed("LINE_NUMBER",                "line_number")
        .withColumnRenamed("QUANTITY",                   "quantity")
        .withColumnRenamed("UNIT_PRICE",                 "unit_price")
        .withColumnRenamed("PRICE",                      "price")
        .withColumn(
            "order_item_discount_amount",
            F.col("ORDER_ITEM_DISCOUNT_AMOUNT").cast(DecimalType(38, 4))
        )
        .drop("ORDER_ITEM_DISCOUNT_AMOUNT")
    )

In [ ]:
# order_detail DROP rules — scaffold placeholder.
# Full rule set + quarantine routing added in the order_detail slice.
ORDER_DETAIL_DROP_RULES: dict[str, str] = {}

OD_CLEAN_PREDICATE   = build_clean_predicate(ORDER_DETAIL_DROP_RULES)   # '1=1'
OD_FAILED_RULES_EXPR = build_failed_rules_expr(ORDER_DETAIL_DROP_RULES)

In [ ]:
@dlt.table(
    name="order_detail",
    comment="Gecleaned order_detail — scaffold; drop/fail/warn regels worden in de volgende slice toegevoegd.",
    table_properties={
        "quality": "silver",
        "pipelines.autoOptimize.managed": "true",
    },
)
def order_detail():
    """order_detail Streaming Table — scaffold."""
    return _read_bronze_order_detail().filter(OD_CLEAN_PREDICATE)

In [ ]:
@dlt.table(
    name="order_detail_quarantine",
    comment="order_detail quarantine — scaffold; drop regels worden in de volgende slice toegevoegd.",
    table_properties={
        "quality": "quarantine",
        "pipelines.autoOptimize.managed": "true",
    },
)
def order_detail_quarantine():
    """order_detail quarantine Streaming Table — scaffold."""
    return (
        _read_bronze_order_detail()
        .filter(f"NOT ({OD_CLEAN_PREDICATE})")
        .withColumn("failed_rules", OD_FAILED_RULES_EXPR)
    )

## `INTEGRATION.sales_line` — Materialised View

Integrated view: `order_header ⨝ order_detail` on `order_id` — one row per order line.

In [ ]:
@dlt.table(
    name="sales_line",
    comment="Geïntegreerde view: order_header ⨝ order_detail op order_id — één rij per orderregel.",
    table_properties={
        "quality": "silver",
        "pipelines.autoOptimize.managed": "true",
    },
)
def sales_line():
    """Materialised View: order_header joined with order_detail."""
    oh = dlt.read("order_header")
    od = dlt.read("order_detail")

    return (
        oh.alias("h")
        .join(od.alias("d"), on="order_id", how="inner")
        .select(
            # Order header keys
            F.col("h.order_id"),
            F.col("h.truck_id"),
            F.col("h.location_id"),
            F.col("h.customer_id"),
            F.col("h.order_channel"),
            F.col("h.order_ts"),
            F.col("h.served_ts"),
            F.col("h.order_currency"),
            F.col("h.order_amount"),
            F.col("h.order_tax_amount"),
            F.col("h.order_discount_amount"),
            F.col("h.order_total"),
            F.col("h.shift_id"),
            F.col("h.shift_start_time"),
            F.col("h.shift_end_time"),
            # Order detail
            F.col("d.order_detail_id"),
            F.col("d.menu_item_id"),
            F.col("d.line_number"),
            F.col("d.quantity"),
            F.col("d.unit_price"),
            F.col("d.price"),
            F.col("d.order_item_discount_amount"),
        )
    )